In [1]:
cd ..

/home/veronika/Documents/2D-lymph-node-vertex-model


In [2]:
import os
import math
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt

import src.auxFunctions as auxFunctions
import src.inputMechanicalParametersModel2 as MechanicalParams2
import src.vertexModel2 as vertexModel2
import src.auxFunctionsExpansion as auxFunctionsExpansion


collision solver could not be imported You may need to install CGAL and re-install tyssue


In [3]:
def collapse_single_edge(cellmap, geom, energyContributions_model, edge_id):
    """
    Collapses a single specified edge by merging its two vertices into one.
    The new vertex is placed at the midpoint of the original edge.
    
    Parameters:
    -----------
    cellmap : object
        The cellmap containing vertex and edge DataFrames
    geom : object
        Geometry handler
    energyContributions_model : object
        Energy model for the system
    edge_id : int
        ID of the edge to collapse (must be an inside edge, not boundary)
    
    Returns:
    --------
    cellmap : object
        Updated cellmap after edge collapse
    """
    
    logger.info(f"Collapsing edge: {edge_id}")
    
    # Verify edge exists
    if edge_id not in cellmap.edge_df.index:
        raise ValueError(f"Edge {edge_id} not found in cellmap.edge_df")
    
    # --- STEP 1: Get the two vertices of the edge ---
    edge_row = cellmap.edge_df.loc[edge_id]
    v1 = edge_row['srce']
    v2 = edge_row['trgt']
    
    logger.info(f"Merging vertices {v1} and {v2}")
    
    # Verify both vertices exist
    if v1 not in cellmap.vert_df.index or v2 not in cellmap.vert_df.index:
        raise ValueError(f"Vertex {v1} or {v2} not found in cellmap.vert_df")
    
    # --- STEP 2: Calculate midpoint coordinates ---
    x1 = cellmap.vert_df.loc[v1, 'x']
    y1 = cellmap.vert_df.loc[v1, 'y']
    x2 = cellmap.vert_df.loc[v2, 'x']
    y2 = cellmap.vert_df.loc[v2, 'y']
    
    midpoint_x = (x1 + x2) / 2
    midpoint_y = (y1 + y2) / 2
    
    # --- STEP 3: Create new vertex at midpoint ---
    new_vertex_id = max(cellmap.vert_df.index) + 1 if not cellmap.vert_df.empty else 0
    new_vertex_data = cellmap.vert_df.loc[v1].copy()
    new_vertex_data['x'] = midpoint_x
    new_vertex_data['y'] = midpoint_y
    cellmap.vert_df.loc[new_vertex_id] = new_vertex_data
    
    # --- STEP 4: Rewire edges - replace v1 and v2 with new vertex ---
    cellmap.edge_df.loc[cellmap.edge_df["srce"] == v1, "srce"] = new_vertex_id
    cellmap.edge_df.loc[cellmap.edge_df["srce"] == v2, "srce"] = new_vertex_id
    cellmap.edge_df.loc[cellmap.edge_df["trgt"] == v1, "trgt"] = new_vertex_id
    cellmap.edge_df.loc[cellmap.edge_df["trgt"] == v2, "trgt"] = new_vertex_id
    
    # --- STEP 5: Delete the collapsed edge and its parallel edge ---
    parallel_edges = cellmap.edge_df[
        ((cellmap.edge_df["srce"] == v1) & (cellmap.edge_df["trgt"] == v2)) |
        ((cellmap.edge_df["srce"] == v2) & (cellmap.edge_df["trgt"] == v1))
    ].index.tolist()
    
    edges_to_delete = [edge_id] + parallel_edges
    cellmap.edge_df.drop(edges_to_delete, inplace=True, errors='ignore')
    
    # --- STEP 6: Delete old vertices ---
    cellmap.vert_df.drop([v1, v2], inplace=True, errors='ignore')
    
    # --- STEP 7: Remove self-loops ---
    cellmap.edge_df = cellmap.edge_df[cellmap.edge_df["srce"] != cellmap.edge_df["trgt"]]
    
    # --- STEP 8: Remove duplicate edges ---
    cellmap.edge_df = cellmap.edge_df.drop_duplicates(subset=['srce', 'trgt'])
    
    # --- STEP 9: Reset indices and update geometry ---
    cellmap.reset_index()
    
    # --- STEP 10: Update active vertices ---
    if hasattr(cellmap, 'active_verts'):
        cellmap.active_verts = list(cellmap.vert_df.index)
    
    geom.update_all(cellmap)
    
    # --- STEP 11: Recompute energy and relax the system ---
    energyContributions_model.compute_energy(cellmap)
    [cellmap, geom, model_H, history_H, solver] = vertexModel2.solveEuler(
        cellmap, geom, energyContributions_model, endTime=40
    )
    
    return cellmap

In [4]:
def split_vertex(cellmap, chosen_vertex, geom, energyContributions_model, distance, retry_attempts=3):
    """
    Safely divide a vertex by creating a nearby new vertex and rewiring edges.
    On failure, rollback and retry up to `retry_attempts` times.

    Returns:
        (cellmap, chosen_vertex, new_vert_index, new_edge_index, opposite_edge_index)
        or (cellmap, chosen_vertex, None, None, None) if all attempts fail.
    """

    original_vert_df = cellmap.vert_df.copy()
    original_edge_df = cellmap.edge_df.copy()

    new_vert_index = None
    new_edge_index = None
    opposite_edge_index = None

    attempt = 0
    while attempt < retry_attempts:
        try:
            # --- Work on temporary copies ---
            temp_vert_df = cellmap.vert_df.copy()
            temp_edge_df = cellmap.edge_df.copy()

            # 1) Find connected edges
            connected_edges = temp_edge_df[
                (temp_edge_df['srce'] == chosen_vertex) |
                (temp_edge_df['trgt'] == chosen_vertex)
            ].copy()

            # 2) Create new vertex nearby
            new_vert_data = temp_vert_df.loc[chosen_vertex].copy()
            angle = np.random.uniform(0, 2*np.pi)
            dx = distance * np.cos(angle)
            dy = distance * np.sin(angle)
            new_vert_data[cellmap.coords] = temp_vert_df.loc[chosen_vertex, cellmap.coords] + [dx, dy]

            new_vert_index = int(temp_vert_df.index.max()) + 1
            temp_vert_df.loc[new_vert_index] = new_vert_data
    
        
            # 3) Create a pair of edges
            # Template: first connected edge (copies all mechanical properties)

            source_edge = connected_edges.iloc[0]

            template = source_edge.copy()

            new_edge_index = int(temp_edge_df.index.max()) + 1
            opposite_edge_index = new_edge_index + 1

            new_edge = template.copy()
            new_edge["srce"], new_edge["trgt"] = chosen_vertex, new_vert_index
            new_edge["face"] = np.nan

            opposite_edge = template.copy()
            opposite_edge["srce"], opposite_edge["trgt"] = new_vert_index, chosen_vertex
            opposite_edge["face"] = np.nan

            temp_edge_df.loc[new_edge_index] = new_edge
            temp_edge_df.loc[opposite_edge_index] = opposite_edge

            # 4) Reassign original edges to closer vertex
            chosen_xy = temp_vert_df.loc[chosen_vertex, cellmap.coords].values.astype(float)
            new_xy    = temp_vert_df.loc[new_vert_index, cellmap.coords].values.astype(float)

            reassigned_edges_to_new_vertex = []
            for e_idx, e in connected_edges.iterrows():
                if e_idx in (new_edge_index, opposite_edge_index):
                    continue
                other = e['trgt'] if e['srce'] == chosen_vertex else e['srce']
                other_xy = temp_vert_df.loc[other, cellmap.coords].values.astype(float)

                d_chosen = np.linalg.norm(other_xy - chosen_xy)
                d_new    = np.linalg.norm(other_xy - new_xy)

                if d_new < d_chosen:
                    reassigned_edges_to_new_vertex.append(e_idx)
                    if e['srce'] == chosen_vertex:
                        temp_edge_df.loc[e_idx, 'srce'] = new_vert_index
                    else:
                        temp_edge_df.loc[e_idx, 'trgt'] = new_vert_index

            # 5) Identify "open" faces (whose edge chains don't close)
            open_faces = []
            for face_id, group in temp_edge_df.groupby('face'):
                verts = list(group[['srce', 'trgt']].itertuples(index=False, name=None))
                if not verts:
                    continue

                # try to follow the loop
                chain = [verts[0][0], verts[0][1]]
                used = {0}
                while True:
                    extended = False
                    for i, (s, t) in enumerate(verts):
                        if i in used:
                            continue
                        if chain[-1] == s:
                            chain.append(t)
                            used.add(i)
                            extended = True
                            break
                        elif chain[-1] == t:
                            chain.append(s)
                            used.add(i)
                            extended = True
                            break
                    if not extended:
                        break

                # if loop doesn't close, this face is "open"
                if chain[0] != chain[-1]:
                    open_faces.append(face_id)

            # filter: only faces touching the new vertex
            open_faces_touching_new = []
            for f in open_faces:
                verts_f = temp_edge_df[temp_edge_df['face'] == f][['srce', 'trgt']].values.ravel()
                if new_vert_index in verts_f or chosen_vertex in verts_f:
                    open_faces_touching_new.append(f)

            print("Open faces touching new vertex:", open_faces_touching_new)

            if len(open_faces_touching_new) != 2:
                raise ValueError(
                    f"Expected 2 open faces, found {len(open_faces_touching_new)}: {open_faces_touching_new}"
                )

            # --- Find which new edge closes which open face ---
            for f in open_faces_touching_new:
                f_edges = temp_edge_df[temp_edge_df['face'] == f]

                # collect src/trgt vertices
                srces = list(f_edges['srce'].astype(int))
                trgts = list(f_edges['trgt'].astype(int))

                # imbalance: start and end
                start_candidates = [v for v in srces if v not in trgts]
                end_candidates   = [v for v in trgts if v not in srces]

                if len(start_candidates) == 1 and len(end_candidates) == 1:
                    start = start_candidates[0]
                    end   = end_candidates[0]
                    needed_edge = (end, start)  # must go end→start to close loop

                    new_pair      = (int(temp_edge_df.loc[new_edge_index, 'srce']),
                                     int(temp_edge_df.loc[new_edge_index, 'trgt']))
                    opposite_pair = (int(temp_edge_df.loc[opposite_edge_index, 'srce']),
                                     int(temp_edge_df.loc[opposite_edge_index, 'trgt']))

                    if new_pair == needed_edge:
                        temp_edge_df.loc[new_edge_index, 'face'] = f
                    elif opposite_pair == needed_edge:
                        temp_edge_df.loc[opposite_edge_index, 'face'] = f
                    else:
                        print(f"Face {f}: expected {needed_edge}, "
                              f"but new={new_pair}, opp={opposite_pair}")
                        
            # --- 6) Verify both assigned faces are closed directed cycles; if not, swap once and recheck

            def _face_closes(df, face_id):
                sub = df[df['face'] == face_id][['srce', 'trgt']]
                # in==out at every vertex
                outc = sub['srce'].value_counts()
                inc  = sub['trgt'].value_counts()
                verts = set(outc.index) | set(inc.index)
                for v in verts:
                    if outc.get(v, 0) != inc.get(v, 0):
                        return False
                # follow edges as a walk using srce->trgt
                start = int(sub.iloc[0]['srce'])
                cur = start
                used = set()
                for _ in range(len(sub)):
                    nxt = sub[~sub.index.isin(used) & (sub['srce'] == cur)]
                    if nxt.empty:
                        return False
                    eidx = nxt.index[0]
                    used.add(eidx)
                    cur = int(sub.loc[eidx, 'trgt'])
                return cur == start and len(used) == len(sub)

            face_new = int(temp_edge_df.loc[new_edge_index, 'face'])
            face_opp = int(temp_edge_df.loc[opposite_edge_index, 'face'])

            ok_new = _face_closes(temp_edge_df, face_new)
            ok_opp = _face_closes(temp_edge_df, face_opp)

            if not (ok_new and ok_opp):
                # try swapping faces between the two new edges once
                temp_edge_df.loc[new_edge_index, 'face'], temp_edge_df.loc[opposite_edge_index, 'face'] = face_opp, face_new
                face_new, face_opp = face_opp, face_new
                ok_new = _face_closes(temp_edge_df, face_new)
                ok_opp = _face_closes(temp_edge_df, face_opp)

            if not (ok_new and ok_opp):
                raise ValueError(
                    f"Face closure failed after assignment: "
                    f"new→face {face_new} ok={ok_new}, opp→face {face_opp} ok={ok_opp}"
            )

            # 7) Commit temp results ---
            cellmap.vert_df = temp_vert_df
            cellmap.edge_df = temp_edge_df

            # Update geometry
            geom.update_all(cellmap)
            cellmap.reset_topo()
            cellmap.reset_index()

            # --- ADD ENERGY RELAXATION HERE ---
            energyContributions_model.compute_energy(cellmap)
            [cellmap, geom, model_H, history_H, solver] = vertexModel2.solveEuler(
                cellmap, geom, energyContributions_model, endTime=40
            )

            print(f" Successfully divided vertex {chosen_vertex} → new vertex {new_vert_index}")
            return cellmap, chosen_vertex, new_vert_index, new_edge_index, opposite_edge_index

        except Exception as e:
            print(f" split_vertex attempt {attempt+1}/{retry_attempts} failed: {e}")
            # rollback original state
            cellmap.vert_df = original_vert_df.copy()
            cellmap.edge_df = original_edge_df.copy()
            geom.update_all(cellmap)
            cellmap.reset_topo()
            cellmap.reset_index()
            attempt += 1

    print("Failed to divide after multiple attempts.")
    return cellmap, chosen_vertex, None, None, None


In [5]:
def run_expansion_simulation_different_fraction_relaxation(
    cellmap_start,
    geom,
    energyContributions_model,
    # Features to include in expansion
    enable_detachment=False,
    enable_divisions=False,
    enable_collapses=False,
    # Directory for saving 
    output_dir="expansion_simulation",
    # Time parameters
    total_steps=15000,
    steps_per_cycle=500,
    # Collapse parameters (apoptosis)
    collapse_fraction_per_remodel=0.0005,
    relax_after_collapse=10,
    # Division parameters
    edge_sum_threshold=3.5,
    division_distance=0.01,
    relax_after_division=10,
    # Factor for line tension relaxation
    tension_decrease_factor=0.4,
    # Fraction of branches to relax
    relax_tension_fraction=0.5,
    # Pressure and elasticity
    pressure_increase_factor=8,
    area_elasticity_value=25,
    # Capsule remodelling to allow expansion
    capsule_tension=300,
    capsule_viscosity=10000,
    # Limit cell cycle events for the inside, to avoid boundary effect
    boundary_layers=5,
    # Plotting
    xlim=(-30, 70),
    ylim=(-30, 70),
):
    """
    Run tissue expansion simulation with optional:
    - Detachment (ECM's preferred length is reset to each edge's actual length)
    - Divisions (cell proliferation with lineage tracking)
    - Collapses (apoptosis/cell contraction)
    - Random tension relaxation (spatially heterogeneous relaxing of branches)
    """

    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, "simulation_stats.json")

    # Build simulation description
    features = []
    if enable_detachment:
        features.append("detachment")
    if enable_divisions:
        features.append(f"divisions_thr{edge_sum_threshold}")
    if enable_collapses:
        features.append(f"collapses_{collapse_fraction_per_remodel:.4f}")
    
    if relax_tension_fraction > 0:
        features.append(f"relax_{int(relax_tension_fraction*100)}%_edges")
    
    simulation_desc = " + ".join(features) if features else "basic"

    cellmap = cellmap_start.copy()

    # ============================================================
    # VERTEX TRACKING (lineage)
    # ============================================================
    v = cellmap.vert_df
    if "new_vert_id" not in v.columns:
        v["new_vert_id"] = np.arange(len(v), dtype=int)
    if "parent_vert_id" not in v.columns:
        v["parent_vert_id"] = np.nan
    if "birth_step" not in v.columns:
        v["birth_step"] = 0
    if "divided_step" not in v.columns:
        v["divided_step"] = np.nan
    next_new_vert_id = int(v["new_vert_id"].max()) + 1

    # ============================================================
    # INITIAL BOUNDARY SETUP (capsule)
    # ============================================================
    boundary_edges, boundary_faces, inside_edges, outside_edges, \
        inside_faces, inside_vertices, outside_vertices = \
        auxFunctions.identify_boundary_layers(cellmap, 1)

    cellmap.vert_df.loc[outside_vertices, "viscosity"] = capsule_viscosity
    cellmap.edge_df.loc[outside_edges, "line_tension"] = capsule_tension

    # ============================================================
    # RANDOM TENSION RELAXATION ("branches relaxing")
    # ============================================================
    edf = cellmap.edge_df
    n_edges = len(edf)
    k = int(np.floor(relax_tension_fraction * n_edges))
    k = max(0, min(k, n_edges))

    if k > 0:
        chosen_edges = np.random.choice(edf.index.to_numpy(), size=k, replace=False)
        edf.loc[chosen_edges, "line_tension"] *= tension_decrease_factor
        print(f"Relaxed {k}/{n_edges} edges to {tension_decrease_factor:.2f}× original tension")

    # ============================================================
    # PRESSURE INCREASE & AREA ELASTICITY
    # ============================================================
    cellmap.face_df["prefered_area"] *= pressure_increase_factor
    cellmap.face_df["area_elasticity"] = area_elasticity_value

    # ============================================================
    # DETACHMENT (ECM loses preferred length)
    # ============================================================
    if enable_detachment:
        cellmap.edge_df["prefered_length"] = cellmap.edge_df["length"]

    # ============================================================
    # STATISTICS LISTS
    # ============================================================
    division_stats = []
    collapse_stats = []
    remodel_nverts_stats = []
    collapse_target_stats = []
    inside_edges_total_stats = []

    # ============================================================
    # INITIAL SAVES
    # ============================================================
    print(f"\n{'='*60}")
    print(f"Starting simulation: {simulation_desc}")
    print(f"Output directory: {output_dir}")
    print(f"{'='*60}\n")

    stats_data = {
        "simulation_description": simulation_desc,
        "parameters": {
            "total_steps": total_steps,
            "steps_per_cycle": steps_per_cycle,
            "collapse_fraction_per_remodel": collapse_fraction_per_remodel,
            "edge_sum_threshold": edge_sum_threshold,
            "division_distance": division_distance,
            "relax_after_collapse": relax_after_collapse,
            "relax_after_division": relax_after_division,
            "tension_decrease_factor": tension_decrease_factor,
            "relax_tension_fraction": relax_tension_fraction,
            "pressure_increase_factor": pressure_increase_factor,
            "area_elasticity_value": area_elasticity_value,
            "capsule_tension": capsule_tension,
            "capsule_viscosity": capsule_viscosity,
            "boundary_layers": boundary_layers,
        },
        "enabled_features": {
            "detachment": enable_detachment,
            "divisions": enable_divisions,
            "collapses": enable_collapses,
        },
    }

    with open(json_path, "w") as f:
        json.dump(stats_data, f, indent=2)

    # Save initial checkpoint
    with open(os.path.join(output_dir, "checkpoint_000000.pkl"), "wb") as f:
        pickle.dump(cellmap, f)

    # Save initial visualization
    fig, ax = auxFunctions.view(cellmap, geom, show_axes=True, xlim=xlim, ylim=ylim)
    plt.title(f"Step 0 | {simulation_desc}", fontsize=10)
    plt.savefig(os.path.join(output_dir, "progress_000000.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

    # ============================================================
    # HELPER FUNCTIONS
    # ============================================================
    def relax(cellmap, nsteps: int):
        """Run mechanical relaxation without topological changes."""
        if nsteps <= 0:
            return cellmap
        energyContributions_model.compute_energy(cellmap)
        cellmap_out, _, _, _, _ = vertexModel2.solveEuler(
            cellmap, geom, energyContributions_model, int(nsteps)
        )
        return cellmap_out

    def ensure_new_vert_ids(cellmap, next_id):
        """Assign unique IDs to any new vertices."""
        vdf = cellmap.vert_df
        if "new_vert_id" not in vdf.columns:
            vdf["new_vert_id"] = np.nan
        missing = vdf["new_vert_id"].isna()
        if missing.any():
            n = int(missing.sum())
            vdf.loc[missing, "new_vert_id"] = np.arange(next_id, next_id + n, dtype=int)
            next_id += n
        return next_id

    # ============================================================
    # MAIN SIMULATION LOOP
    # ============================================================
    for step in range(0, total_steps, steps_per_cycle):
        current_step = step + steps_per_cycle
        print(f"\n--- Cycle {current_step}/{total_steps} ---")

        # Re-identify boundaries
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, boundary_layers)

        current_divisions = 0
        current_collapses = 0

        # --------------------------------------------------------
        # 1. COLLAPSES (apoptosis) - USING collapse_single_edge
        # --------------------------------------------------------
        if enable_collapses:
            valid_inside_edges = [e for e in inside_edges if e in cellmap.edge_df.index]
            n_inside = len(valid_inside_edges)
            target_collapse = int(math.floor(collapse_fraction_per_remodel * n_inside))
            target_collapse = max(0, min(target_collapse, n_inside))

            inside_edges_total_stats.append(n_inside)
            collapse_target_stats.append(target_collapse)

            if target_collapse > 0:
                edges_to_collapse = np.random.choice(valid_inside_edges, size=target_collapse, replace=False)
                for edge in edges_to_collapse:
                    try:
                        # USING YOUR collapse_single_edge FUNCTION
                        cellmap = collapse_single_edge(
                            cellmap, geom, energyContributions_model, edge
                        )
                        current_collapses += 1
                    except Exception as e:
                        print(f"  Warning: collapse failed for edge {edge}: {e}")

            print(f"  Collapses: {current_collapses}/{target_collapse}")

            cellmap.reset_index()
            cellmap.reset_topo()
            next_new_vert_id = ensure_new_vert_ids(cellmap, next_new_vert_id)

            if relax_after_collapse > 0:
                cellmap = relax(cellmap, relax_after_collapse)
                cellmap.reset_index()
                cellmap.reset_topo()

        # Re-identify boundaries after collapses
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, boundary_layers)

        # --------------------------------------------------------
        # 2. DIVISIONS (cell proliferation) - USING split_vertex
        # --------------------------------------------------------
        if enable_divisions:
            # Find vertices to divide using edge_sum_threshold
            vertices_to_divide = []
            for v in inside_vertices:
                if v not in cellmap.vert_df.index:
                    continue
                # Calculate sum of connected edge lengths
                connected_edges = cellmap.edge_df[
                    (cellmap.edge_df["srce"] == v) | (cellmap.edge_df["trgt"] == v)
                ]
                edge_sum = 0
                for _, row in connected_edges.iterrows():
                    # Get edge length from geometry
                    length = geom.length(
                        cellmap.vert_df.loc[row["srce"]],
                        cellmap.vert_df.loc[row["trgt"]]
                    )
                    edge_sum += length
                
                if edge_sum > edge_sum_threshold:
                    vertices_to_divide.append(v)
            
            remodel_nverts_stats.append(len(vertices_to_divide))
            print(f"  Vertices above threshold: {len(vertices_to_divide)}")

            if len(vertices_to_divide) > 0:
                # Limit divisions per cycle to avoid performance issues
                max_divisions_per_cycle = 20
                divided_count = 0
                
                for v in vertices_to_divide[:max_divisions_per_cycle]:
                    try:
                        # USING YOUR split_vertex FUNCTION
                        cellmap, parent_vert, new_vert, new_edge, opp_edge = split_vertex(
                            cellmap, v, geom, energyContributions_model, division_distance, retry_attempts=3
                        )
                        
                        if new_vert is not None:
                            # Record lineage
                            parent_id = int(cellmap.vert_df.loc[parent_vert, "new_vert_id"])
                            cellmap.vert_df.loc[parent_vert, "divided_step"] = current_step
                            cellmap.vert_df.loc[new_vert, "new_vert_id"] = next_new_vert_id
                            cellmap.vert_df.loc[new_vert, "parent_vert_id"] = parent_id
                            cellmap.vert_df.loc[new_vert, "birth_step"] = current_step
                            next_new_vert_id += 1
                            divided_count += 1
                            
                    except Exception as e:
                        print(f"  Warning: division failed for vertex {v}: {e}")
                
                current_divisions = divided_count

                # Save post-division checkpoint
                with open(os.path.join(output_dir, f"checkpoint_{current_step:06d}_postdiv.pkl"), "wb") as f:
                    pickle.dump(cellmap, f)

            print(f"  Divisions: {current_divisions}")

            if relax_after_division > 0:
                cellmap = relax(cellmap, relax_after_division)
                cellmap.reset_index()
                cellmap.reset_topo()

        # --------------------------------------------------------
        # 3. MAIN RELAXATION TO END CYCLE
        # --------------------------------------------------------
        used_steps = 0
        if enable_collapses:
            used_steps += relax_after_collapse
        if enable_divisions:
            used_steps += relax_after_division

        remaining_steps = steps_per_cycle - used_steps
        if remaining_steps < 0:
            raise ValueError("relax_after_collapse + relax_after_division exceeds steps_per_cycle")

        if remaining_steps > 0:
            cellmap = relax(cellmap, remaining_steps)

        # --------------------------------------------------------
        # 4. RECORD STATISTICS
        # --------------------------------------------------------
        division_stats.append(current_divisions)
        collapse_stats.append(current_collapses)

        # --------------------------------------------------------
        # 5. SAVE CHECKPOINT & VISUALIZATION
        # --------------------------------------------------------
        with open(os.path.join(output_dir, f"checkpoint_{current_step:06d}.pkl"), "wb") as f:
            pickle.dump(cellmap, f)

        fig, ax = auxFunctions.view(cellmap, geom, show_axes=True, xlim=xlim, ylim=ylim)
        title = f"Step {current_step} | {simulation_desc} | Div:{current_divisions} Col:{current_collapses}"
        plt.title(title, fontsize=10)
        plt.savefig(os.path.join(output_dir, f"progress_{current_step:06d}.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)

        # Update JSON statistics
        with open(json_path, "r") as f:
            existing_data = json.load(f)

        existing_data["last_saved_step"] = current_step
        existing_data["statistics"] = {
            "division_stats": division_stats if enable_divisions else None,
            "collapse_stats": collapse_stats if enable_collapses else None,
            "remodel_nverts_stats": remodel_nverts_stats,
            "collapse_target_stats": collapse_target_stats if enable_collapses else None,
            "inside_edges_total_stats": inside_edges_total_stats if enable_collapses else None,
        }

        with open(json_path, "w") as f:
            json.dump(existing_data, f, indent=2)

    return cellmap, division_stats, collapse_stats

In [6]:
# Initialize cellmap, geometry, and energy contributions model
cellmap_init, geom, energyContributions_model = vertexModel2.initialize(40)
cellmap_init = MechanicalParams2.update(cellmap_init)

boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(cellmap_init, 1)

#Change outside vertices mechanics
high_viscosity_value = 1000000  # A very large viscosity value
for vertex_id in outside_vertices:
    if vertex_id in cellmap_init.vert_df.index:
        cellmap_init.vert_df.at[vertex_id, "viscosity"] = high_viscosity_value

        
#energyContributions_model.compute_energy(cellmap)
energyContributions_model.compute_energy(cellmap_init)
[cellmap_init, geom, energyContributions_model, history_new, solver1] = vertexModel2.solveEuler(cellmap_init, geom, energyContributions_model, 20)


CGAL-based mesh generation utilities not found, you may need to install CGAL and build from source
C++ extensions are not available for this version
Topology changed!


In [32]:
cellmap, division_stats, collapse_stats = run_expansion_simulation_different_fraction_relaxation(
    cellmap_init,
    geom,
    energyContributions_model,
    # Features to include in expansion
    enable_detachment=False,
    enable_divisions=False,
    enable_collapses=False,
    # Directory for saving 
    output_dir="expansion_simulation_test",
    # Time parameters
    total_steps=15000,
    steps_per_cycle=500,
    # Collapse parameters (apoptosis)
    collapse_fraction_per_remodel=0.0005,
    relax_after_collapse=10,
    # Division parameters
    edge_sum_threshold=3.5,
    division_distance=0.01,
    relax_after_division=10,
    # Factor for line tension relaxation
    tension_decrease_factor=0.4,
    # Fraction of branches to relax
    relax_tension_fraction=0.5,
    # Pressure and elasticity
    pressure_increase_factor=8,
    area_elasticity_value=25,
    # Capsule remodelling to allow expansion
    capsule_tension=300,
    capsule_viscosity=10000,
    # Limit cell cycle events for the inside, to avoid boundary effect
    boundary_layers=5,
    # Plotting
    xlim=(-30, 70),
    ylim=(-30, 70),
)

Relaxed 4065/8130 edges to 0.40× original tension

Starting simulation: relax_50%_edges
Output directory: expansion_simulation_test


--- Cycle 500/15000 ---

--- Cycle 1000/15000 ---

--- Cycle 1500/15000 ---

--- Cycle 2000/15000 ---

--- Cycle 2500/15000 ---

--- Cycle 3000/15000 ---

--- Cycle 3500/15000 ---

--- Cycle 4000/15000 ---


KeyboardInterrupt: 

In [7]:
### now relaxing branhces :

def run_expansion_simulation_different_fraction_relaxation(
    cellmap_start,
    geom,
    energyContributions_model,
    # Features to include in expansion
    enable_detachment=False,
    enable_divisions=False,
    enable_collapses=False,
    # Directory for saving 
    output_dir="expansion_simulation",
    # Time parameters
    total_steps=15000,
    steps_per_cycle=500,
    # Collapse parameters (apoptosis)
    collapse_fraction_per_remodel=0.0005,
    relax_after_collapse=10,
    # Division parameters
    edge_sum_threshold=3.5,
    division_distance=0.01,
    relax_after_division=10,
    # Factor for line tension relaxation
    tension_decrease_factor=0.4,
    # Fraction of BRANCHES (physical edges) to relax
    relax_tension_fraction=0.5,
    # Pressure and elasticity
    pressure_increase_factor=8,
    area_elasticity_value=25,
    # Capsule remodelling to allow expansion
    capsule_tension=300,
    capsule_viscosity=10000,
    # Limit cell cycle events for the inside, to avoid boundary effect
    boundary_layers=5,
    # Plotting
    xlim=(-30, 70),
    ylim=(-30, 70),
):
    """
    Run tissue expansion simulation with optional:
    - Detachment (ECM's preferred length is reset to each edge's actual length)
    - Divisions (cell proliferation with lineage tracking)
    - Collapses (apoptosis/cell contraction)
    - Random tension relaxation on UNDIRECTED EDGE PAIRS (whole branches)
    
    Parameters
    ----------
    relax_tension_fraction : float, default=0.5
        Fraction of PHYSICAL EDGES (undirected pairs) randomly selected for tension reduction.
        Both half-edges of a selected physical edge are relaxed together.
        Use 0.5, 0.7, 0.9, 0.99, 1.0 for "branches relaxed" experiments.
    
    tension_decrease_factor : float, default=0.4
        Factor to multiply line_tension by for selected edges.
        0.4 = reduce to 40% of original tension.
    """

    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, "simulation_stats.json")

    # Build simulation description
    features = []
    if enable_detachment:
        features.append("detachment")
    if enable_divisions:
        features.append(f"divisions_thr{edge_sum_threshold}")
    if enable_collapses:
        features.append(f"collapses_{collapse_fraction_per_remodel:.4f}")
    
    if relax_tension_fraction > 0:
        features.append(f"relax_{int(relax_tension_fraction*100)}%_branches")
    
    simulation_desc = " + ".join(features) if features else "basic"

    cellmap = cellmap_start.copy()

    # ============================================================
    # VERTEX TRACKING (lineage)
    # ============================================================
    v = cellmap.vert_df
    if "new_vert_id" not in v.columns:
        v["new_vert_id"] = np.arange(len(v), dtype=int)
    if "parent_vert_id" not in v.columns:
        v["parent_vert_id"] = np.nan
    if "birth_step" not in v.columns:
        v["birth_step"] = 0
    if "divided_step" not in v.columns:
        v["divided_step"] = np.nan
    next_new_vert_id = int(v["new_vert_id"].max()) + 1

    # ============================================================
    # INITIAL BOUNDARY SETUP (capsule)
    # ============================================================
    boundary_edges, boundary_faces, inside_edges, outside_edges, \
        inside_faces, inside_vertices, outside_vertices = \
        auxFunctions.identify_boundary_layers(cellmap, 1)

    cellmap.vert_df.loc[outside_vertices, "viscosity"] = capsule_viscosity
    cellmap.edge_df.loc[outside_edges, "line_tension"] = capsule_tension

    # ============================================================
    # RANDOM TENSION RELAXATION ON UNDIRECTED EDGE PAIRS
    # ("branches relaxing" - BOTH half-edges together)
    # ============================================================
    edf = cellmap.edge_df
    
    # First, find all unique undirected physical edges
    # Group half-edges by their undirected key (min_srce, max_trgt)
    physical_edges = {}
    for idx, row in edf.iterrows():
        srce = row['srce']
        trgt = row['trgt']
        key = (min(srce, trgt), max(srce, trgt))
        if key not in physical_edges:
            physical_edges[key] = []
        physical_edges[key].append(idx)
    
    # Count unique physical edges (each has 1 or 2 half-edges)
    # Inside edges have 2 half-edges, boundary edges have 1
    physical_edge_list = list(physical_edges.keys())
    n_physical_edges = len(physical_edge_list)
    
    # Calculate how many physical edges to relax
    k_physical = int(np.floor(relax_tension_fraction * n_physical_edges))
    k_physical = max(0, min(k_physical, n_physical_edges))
    
    if k_physical > 0:
        # Randomly select physical edges (by their undirected key)
        chosen_physical_keys = np.random.choice(
            [i for i in range(n_physical_edges)], 
            size=k_physical, 
            replace=False
        )
        
        # For each chosen physical edge, relax BOTH half-edges
        relaxed_half_edge_count = 0
        for key_idx in chosen_physical_keys:
            key = physical_edge_list[key_idx]
            half_edge_ids = physical_edges[key]
            
            # Relax all half-edges belonging to this physical edge
            edf.loc[half_edge_ids, "line_tension"] *= tension_decrease_factor
            relaxed_half_edge_count += len(half_edge_ids)
        
        print(f"Relaxed {k_physical}/{n_physical_edges} physical edges")
        print(f"  → Affected {relaxed_half_edge_count} half-edges")
        print(f"  → Tension reduced to {tension_decrease_factor:.2f}× original")
    
    # ============================================================
    # PRESSURE INCREASE & AREA ELASTICITY
    # ============================================================
    cellmap.face_df["prefered_area"] *= pressure_increase_factor
    cellmap.face_df["area_elasticity"] = area_elasticity_value

    # ============================================================
    # DETACHMENT (ECM loses preferred length)
    # ============================================================
    if enable_detachment:
        cellmap.edge_df["prefered_length"] = cellmap.edge_df["length"]

    # ============================================================
    # STATISTICS LISTS
    # ============================================================
    division_stats = []
    collapse_stats = []
    remodel_nverts_stats = []
    collapse_target_stats = []
    inside_edges_total_stats = []

    # ============================================================
    # INITIAL SAVES
    # ============================================================
    print(f"\n{'='*60}")
    print(f"Starting simulation: {simulation_desc}")
    print(f"Output directory: {output_dir}")
    print(f"{'='*60}\n")

    stats_data = {
        "simulation_description": simulation_desc,
        "parameters": {
            "total_steps": total_steps,
            "steps_per_cycle": steps_per_cycle,
            "collapse_fraction_per_remodel": collapse_fraction_per_remodel,
            "edge_sum_threshold": edge_sum_threshold,
            "division_distance": division_distance,
            "relax_after_collapse": relax_after_collapse,
            "relax_after_division": relax_after_division,
            "tension_decrease_factor": tension_decrease_factor,
            "relax_tension_fraction": relax_tension_fraction,
            "pressure_increase_factor": pressure_increase_factor,
            "area_elasticity_value": area_elasticity_value,
            "capsule_tension": capsule_tension,
            "capsule_viscosity": capsule_viscosity,
            "boundary_layers": boundary_layers,
        },
        "enabled_features": {
            "detachment": enable_detachment,
            "divisions": enable_divisions,
            "collapses": enable_collapses,
        },
    }

    with open(json_path, "w") as f:
        json.dump(stats_data, f, indent=2)

    # Save initial checkpoint
    with open(os.path.join(output_dir, "checkpoint_000000.pkl"), "wb") as f:
        pickle.dump(cellmap, f)

    # Save initial visualization
    fig, ax = auxFunctions.view(cellmap, geom, show_axes=True, xlim=xlim, ylim=ylim)
    plt.title(f"Step 0 | {simulation_desc}", fontsize=10)
    plt.savefig(os.path.join(output_dir, "progress_000000.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

    # ============================================================
    # HELPER FUNCTIONS
    # ============================================================
    def relax(cellmap, nsteps: int):
        """Run mechanical relaxation without topological changes."""
        if nsteps <= 0:
            return cellmap
        energyContributions_model.compute_energy(cellmap)
        cellmap_out, _, _, _, _ = vertexModel2.solveEuler(
            cellmap, geom, energyContributions_model, int(nsteps)
        )
        return cellmap_out

    def ensure_new_vert_ids(cellmap, next_id):
        """Assign unique IDs to any new vertices."""
        vdf = cellmap.vert_df
        if "new_vert_id" not in vdf.columns:
            vdf["new_vert_id"] = np.nan
        missing = vdf["new_vert_id"].isna()
        if missing.any():
            n = int(missing.sum())
            vdf.loc[missing, "new_vert_id"] = np.arange(next_id, next_id + n, dtype=int)
            next_id += n
        return next_id

    # ============================================================
    # MAIN SIMULATION LOOP
    # ============================================================
    for step in range(0, total_steps, steps_per_cycle):
        current_step = step + steps_per_cycle
        print(f"\n--- Cycle {current_step}/{total_steps} ---")

        # Re-identify boundaries
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, boundary_layers)

        current_divisions = 0
        current_collapses = 0

        # --------------------------------------------------------
        # 1. COLLAPSES (apoptosis) - USING collapse_single_edge
        # --------------------------------------------------------
        if enable_collapses:
            valid_inside_edges = [e for e in inside_edges if e in cellmap.edge_df.index]
            n_inside = len(valid_inside_edges)
            target_collapse = int(math.floor(collapse_fraction_per_remodel * n_inside))
            target_collapse = max(0, min(target_collapse, n_inside))

            inside_edges_total_stats.append(n_inside)
            collapse_target_stats.append(target_collapse)

            if target_collapse > 0:
                edges_to_collapse = np.random.choice(valid_inside_edges, size=target_collapse, replace=False)
                for edge in edges_to_collapse:
                    try:
                        cellmap = collapse_single_edge(
                            cellmap, geom, energyContributions_model, edge
                        )
                        current_collapses += 1
                    except Exception as e:
                        print(f"  Warning: collapse failed for edge {edge}: {e}")

            print(f"  Collapses: {current_collapses}/{target_collapse}")

            cellmap.reset_index()
            cellmap.reset_topo()
            next_new_vert_id = ensure_new_vert_ids(cellmap, next_new_vert_id)

            if relax_after_collapse > 0:
                cellmap = relax(cellmap, relax_after_collapse)
                cellmap.reset_index()
                cellmap.reset_topo()

        # Re-identify boundaries after collapses
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, boundary_layers)

        # --------------------------------------------------------
        # 2. DIVISIONS (cell proliferation) - USING split_vertex
        # --------------------------------------------------------
        if enable_divisions:
            # Find vertices to divide using edge_sum_threshold
            vertices_to_divide = []
            for v in inside_vertices:
                if v not in cellmap.vert_df.index:
                    continue
                # Calculate sum of connected edge lengths
                connected_edges = cellmap.edge_df[
                    (cellmap.edge_df["srce"] == v) | (cellmap.edge_df["trgt"] == v)
                ]
                edge_sum = 0
                for _, row in connected_edges.iterrows():
                    length = geom.length(
                        cellmap.vert_df.loc[row["srce"]],
                        cellmap.vert_df.loc[row["trgt"]]
                    )
                    edge_sum += length
                
                if edge_sum > edge_sum_threshold:
                    vertices_to_divide.append(v)
            
            remodel_nverts_stats.append(len(vertices_to_divide))
            print(f"  Vertices above threshold: {len(vertices_to_divide)}")

            if len(vertices_to_divide) > 0:
                max_divisions_per_cycle = 20
                divided_count = 0
                
                for v in vertices_to_divide[:max_divisions_per_cycle]:
                    try:
                        cellmap, parent_vert, new_vert, new_edge, opp_edge = split_vertex(
                            cellmap, v, geom, energyContributions_model, division_distance, retry_attempts=3
                        )
                        
                        if new_vert is not None:
                            parent_id = int(cellmap.vert_df.loc[parent_vert, "new_vert_id"])
                            cellmap.vert_df.loc[parent_vert, "divided_step"] = current_step
                            cellmap.vert_df.loc[new_vert, "new_vert_id"] = next_new_vert_id
                            cellmap.vert_df.loc[new_vert, "parent_vert_id"] = parent_id
                            cellmap.vert_df.loc[new_vert, "birth_step"] = current_step
                            next_new_vert_id += 1
                            divided_count += 1
                            
                    except Exception as e:
                        print(f"  Warning: division failed for vertex {v}: {e}")
                
                current_divisions = divided_count

                with open(os.path.join(output_dir, f"checkpoint_{current_step:06d}_postdiv.pkl"), "wb") as f:
                    pickle.dump(cellmap, f)

            print(f"  Divisions: {current_divisions}")

            if relax_after_division > 0:
                cellmap = relax(cellmap, relax_after_division)
                cellmap.reset_index()
                cellmap.reset_topo()

        # --------------------------------------------------------
        # 3. MAIN RELAXATION TO END CYCLE
        # --------------------------------------------------------
        used_steps = 0
        if enable_collapses:
            used_steps += relax_after_collapse
        if enable_divisions:
            used_steps += relax_after_division

        remaining_steps = steps_per_cycle - used_steps
        if remaining_steps < 0:
            raise ValueError("relax_after_collapse + relax_after_division exceeds steps_per_cycle")

        if remaining_steps > 0:
            cellmap = relax(cellmap, remaining_steps)

        # --------------------------------------------------------
        # 4. RECORD STATISTICS
        # --------------------------------------------------------
        division_stats.append(current_divisions)
        collapse_stats.append(current_collapses)

        # --------------------------------------------------------
        # 5. SAVE CHECKPOINT & VISUALIZATION
        # --------------------------------------------------------
        with open(os.path.join(output_dir, f"checkpoint_{current_step:06d}.pkl"), "wb") as f:
            pickle.dump(cellmap, f)

        fig, ax = auxFunctions.view(cellmap, geom, show_axes=True, xlim=xlim, ylim=ylim)
        title = f"Step {current_step} | {simulation_desc} | Div:{current_divisions} Col:{current_collapses}"
        plt.title(title, fontsize=10)
        plt.savefig(os.path.join(output_dir, f"progress_{current_step:06d}.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)

        # Update JSON statistics
        with open(json_path, "r") as f:
            existing_data = json.load(f)

        existing_data["last_saved_step"] = current_step
        existing_data["statistics"] = {
            "division_stats": division_stats if enable_divisions else None,
            "collapse_stats": collapse_stats if enable_collapses else None,
            "remodel_nverts_stats": remodel_nverts_stats,
            "collapse_target_stats": collapse_target_stats if enable_collapses else None,
            "inside_edges_total_stats": inside_edges_total_stats if enable_collapses else None,
        }

        with open(json_path, "w") as f:
            json.dump(existing_data, f, indent=2)

    return cellmap, division_stats, collapse_stats

In [ ]:
run_expansion_simulation_different_fraction_relaxation(
    cellmap_init,
    geom,
    energyContributions_model,
    # Features to include in expansion
    enable_detachment=False,
    enable_divisions=False,
    enable_collapses=False,
    # Directory for saving 
    output_dir="expansion_simulation_test_branches",
    # Time parameters
    total_steps=15000,
    steps_per_cycle=500,
    # Collapse parameters (apoptosis)
    collapse_fraction_per_remodel=0.0005,
    relax_after_collapse=10,
    # Division parameters
    edge_sum_threshold=3.5,
    division_distance=0.01,
    relax_after_division=10,
    # Factor for line tension relaxation
    tension_decrease_factor=0.4,
    # Fraction of BRANCHES (physical edges) to relax
    relax_tension_fraction=0.5,
    # Pressure and elasticity
    pressure_increase_factor=8,
    area_elasticity_value=25,
    # Capsule remodelling to allow expansion
    capsule_tension=300,
    capsule_viscosity=10000,
    # Limit cell cycle events for the inside, to avoid boundary effect
    boundary_layers=5,
    # Plotting
    xlim=(-30, 70),
    ylim=(-30, 70),
)

Relaxed 2066/4133 physical edges
  → Affected 4060 half-edges
  → Tension reduced to 0.40× original

Starting simulation: relax_50%_branches
Output directory: expansion_simulation_test_branches


--- Cycle 500/15000 ---

--- Cycle 1000/15000 ---

--- Cycle 1500/15000 ---

--- Cycle 2000/15000 ---
